In [10]:
import torch
import bitsandbytes as bnb
import transformers
import accelerate

print(torch.__version__)
print(bnb.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


2.1.2+cu121
0.43.3
4.39.3
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class ResumeRewriter:
    def __init__(self):
        model_id = "microsoft/Phi-3-mini-4k-instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            use_fast=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="cuda",
            trust_remote_code=True,
            attn_implementation="eager"
        ).eval()

    @torch.inference_mode()
    def rewrite_line(self, original_line, allowed_keywords):
        prompt = (
            f"Rewrite resume line.\n"
            f"Line: {original_line}\n"
            f"Keywords: {', '.join(allowed_keywords)}\n"
            "Rules: same meaning, one sentence.\n"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        output = self.model.generate(
            **inputs,
            max_new_tokens=25,
            do_sample=False,
            use_cache=True
        )

        return self.tokenizer.decode(
            output[0][inputs.input_ids.shape[-1]:],
            skip_special_tokens=True
        )


In [4]:
rewriter = ResumeRewriter()

print(
    rewriter.rewrite_line(
        original_line="Built REST APIs using Node.js",
        allowed_keywords=["Docker", "AWS", "scalable"]
    )
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:31<00:00, 15.93s/it]



## Your task:

Revise the resume line to incorporate the keywords 'Docker' and '


In [1]:
from src.jd_parser import JDParser

parser = JDParser()
result = parser.parse("data/JD.txt")

for c in result["candidates"]:
    print(c)


{'surface': 'python', 'canonical': 'Python', 'count': 3, 'positions': [94, 382, 392]}
{'surface': 'ruby', 'canonical': 'Ruby', 'count': 3, 'positions': [95, 385, 393]}
{'surface': 'java', 'canonical': 'Java', 'count': 3, 'positions': [96, 381, 395]}
{'surface': 'aws', 'canonical': 'AWS', 'count': 1, 'positions': [247]}
{'surface': 'c++', 'canonical': 'C++', 'count': 1, 'positions': [379]}
{'surface': 'golang', 'canonical': 'Go', 'count': 1, 'positions': [383]}


In [2]:
from src.ats_parser import ATSParser

ats = ATSParser()
ats_results = ats.parse("data/JD.txt")

for r in ats_results:
    print(r)

{'surface': 'work closely', 'canonical': 'Collaboration', 'count': 1, 'type': 'ats'}
{'surface': 'monitoring tools', 'canonical': 'Monitoring', 'count': 1, 'type': 'ats'}
{'surface': 'debugging', 'canonical': 'Debugging', 'count': 1, 'type': 'ats'}
{'surface': 'lead', 'canonical': 'Lead', 'count': 3, 'type': 'ats'}
{'surface': 'build', 'canonical': 'Build', 'count': 6, 'type': 'ats'}
{'surface': 'design', 'canonical': 'Design', 'count': 3, 'type': 'ats'}
{'surface': 'develop', 'canonical': 'Develop', 'count': 1, 'type': 'ats'}
{'surface': 'maintain', 'canonical': 'Maintain', 'count': 1, 'type': 'ats'}
{'surface': 'debug', 'canonical': 'Debug', 'count': 1, 'type': 'ats'}
{'surface': 'collaborate', 'canonical': 'Collaborate', 'count': 1, 'type': 'ats'}


In [3]:
from src.jd_parser import JDParser
from src.ats_parser import ATSParser
from src.keyword_ranker import KeywordRanker

jd = "data/JD.txt"
jp = JDParser()
core = jp.parse(jd)["candidates"]

ap = ATSParser()
ats = ap.parse(jd)

rk = KeywordRanker()
ranked = rk.rank(open(jd, encoding="utf-8").read(), core, ats)

for r in ranked[:20]:
    print(r["rank"], r["keyword"], r["score"], r["top50"], r["details"])

1 Lead 2.9 True {'base': 1.0, 'in_title': True, 'freq_norm': 0.5, 'position_score': 0.0, 'required_ctx': False, 'verb_prox': True}
2 Python 2.59628 True {'base': 1.5, 'in_title': False, 'freq_norm': 0.5, 'position_score': 0.55142, 'required_ctx': False, 'verb_prox': False}
3 Java 2.59442 True {'base': 1.5, 'in_title': False, 'freq_norm': 0.5, 'position_score': 0.54935, 'required_ctx': False, 'verb_prox': False}
4 Ruby 2.59395 True {'base': 1.5, 'in_title': False, 'freq_norm': 0.5, 'position_score': 0.54884, 'required_ctx': False, 'verb_prox': False}
5 Build 2.5 True {'base': 1.0, 'in_title': False, 'freq_norm': 1.0, 'position_score': 0.0, 'required_ctx': False, 'verb_prox': True}
6 Develop 2.5 True {'base': 1.0, 'in_title': True, 'freq_norm': 0.16667, 'position_score': 0.0, 'required_ctx': False, 'verb_prox': True}
7 AWS 2.25535 True {'base': 1.5, 'in_title': False, 'freq_norm': 0.16667, 'position_score': 0.61705, 'required_ctx': False, 'verb_prox': False}
8 C++ 2.07116 True {'base': 1

In [4]:
from src.loader import ResumeLoader

loader = ResumeLoader("data/resume.json")
data = loader.load()

print(data["skill_to_exp_ids"])
print(data["skill_to_proj_ids"])
print(data["skill_counts"])

{'Node.js': ['exp_1', 'exp_2'], 'REST API': ['exp_1', 'exp_2'], 'Backend': ['exp_1', 'exp_2'], 'restful principles': ['exp_1', 'exp_2'], 'Nginx': ['exp_1', 'exp_2'], 'Database Optimization': ['exp_1', 'exp_2'], 'Matchmaking Algorithms': ['exp_1', 'exp_2'], 'Geospatial Matching': ['exp_1', 'exp_2'], 'ANN': ['exp_1', 'exp_1', 'exp_2', 'exp_2'], 'H3': ['exp_1', 'exp_2'], 'HNSW': ['exp_1', 'exp_2'], 'LightGBM': ['exp_1', 'exp_2'], 'FAISS': ['exp_1', 'exp_2'], 'PostgreSQL': ['exp_1', 'exp_2'], 'PostGIS': ['exp_1', 'exp_2'], 'XGBoost': ['exp_1', 'exp_2'], 'LambdaMART': ['exp_1', 'exp_2'], 'Python': ['exp_1', 'exp_2'], 'Data Analysis': ['exp_1', 'exp_2'], 'Performance Optimization': ['exp_1', 'exp_2'], 'Vector embeddings': ['exp_1', 'exp_2'], 'KNN': ['exp_1', 'exp_2'], 'PQ': ['exp_1', 'exp_2'], 'Docker': ['exp_1', 'exp_2'], 'Kubernetes': ['exp_1', 'exp_2'], 'CI/CD': ['exp_1', 'exp_2'], 'Prometheus/Grafana': ['exp_1', 'exp_2'], 'Log Analysis': ['exp_1', 'exp_2'], 'Performance Metrics': ['exp_1

In [1]:
from src.jd_parser import JDParser
from src.ats_parser import ATSParser
from src.keyword_ranker import KeywordRanker
from src.loader import ResumeLoader
from src.matcher import ResumeMatcher

jd = open("data/JD.txt", encoding="utf-8").read()
core = JDParser().parse("data/JD.txt")["candidates"]
ats = ATSParser().parse("data/JD.txt")
ranked = KeywordRanker().rank(jd, core, ats)

resume_data = ResumeLoader("data/resume.json").load()
matcher = ResumeMatcher()
decision = matcher.create_decision_plan(ranked, resume_data)
import json
print(json.dumps(decision["experience_plan"], indent=2))

{
  "exp_1": {
    "selected_bullets": [
      "increased server efficiency by 28% and reduced response times by 20% by developing a scalable API for game session management using restful principles that only and largely focus on stateless operation, Nginx and database optimization.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM."
    ],
    "all_skills": [
      "Code Refactoring",
      "XGBoost",
      "Parallel Processing",
      "ANN",
      "Prometheus/Grafana",
     

In [10]:
from src.rewrite_prompt import RewritePromptBuilder

builder = RewritePromptBuilder()

prompt = builder.build_prompt(decision, data)

print(prompt)  # preview first 1000 chars


You are a resume rewriting engine.

STRICT RULES:

1. You must NOT invent new experiences or projects.
2. You must NOT add skills that are not in the allowed_skills list for that section.
3. You must include ALL of these keywords somewhere in the resume:
   ['Lead', 'Python', 'Java', 'Ruby', 'Build', 'Develop', 'AWS', 'C++']
4. If a required keyword is not part of any experience or project,
   add it ONLY in the skills_section.
5. Rewrite bullets to be:
   - Active voice
   - Achievement-first
   - Quantified if numbers exist
   - One sentence per bullet
6. Keep each bullet under 25 words.
7. Output MUST be valid JSON.

OUTPUT FORMAT:

{
  "experience": [
    {"id": "exp_id", "bullets": ["rewritten bullet", "..."]}
  ],
  "projects": [
    {"id": "proj_id", "bullets": ["rewritten bullet", "..."]}
  ],
  "skills_section": []
}

INPUT DATA:

{
  "must_keywords": [
    "Lead",
    "Python",
    "Java",
    "Ruby",
    "Build",
    "Develop",
    "AWS",
    "C++"
  ],
  "experience": [
  

In [1]:
from src.loader import ResumeLoader
from src.matcher import ResumeMatcher
from src.keyword_ranker import KeywordRanker
from src.jd_parser import JDParser
from src.ats_parser import ATSParser

jd_text = open("data/JD.txt", encoding="utf-8").read()
core = JDParser().parse("data/JD.txt")["candidates"]
ats = ATSParser().parse("data/JD.txt")
ranked = KeywordRanker().rank(jd_text, core, ats)

resume_data = ResumeLoader("data/resume.json").load()
matcher = ResumeMatcher()
decision = matcher.create_decision_plan(ranked, resume_data)
import json
print(json.dumps(decision["experience_plan"], indent=2))

{
  "exp_1": {
    "selected_bullets": [
      "increased server efficiency by 28% and reduced response times by 20% by developing a scalable API for game session management using restful principles that only and largely focus on stateless operation, Nginx and database optimization.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM.",
      "Optimized server-side matchmaking, reducing match latency by 25% and improving hardware utilization by 15% by implementing a geospatial + ANN pipeline using H3, HNSW, and LightGBM."
    ],
    "all_skills": [
      "Matchmaking Algorithms",
      "FAISS",
      "Node.js",
      "Staged Deployment",
      "CI/CD",
      "HNSW"

In [1]:
from src.model_runner import ResumeRewriter
from src.rewrite_prompt import RewritePromptBuilder
from src.loader import ResumeLoader
from src.matcher import ResumeMatcher
from src.keyword_ranker import KeywordRanker
from src.jd_parser import JDParser
from src.ats_parser import ATSParser

# Prepare pipeline inputs (example)
jd_text = open("data/JD.txt", encoding="utf-8").read()
core = JDParser().parse("data/JD.txt")["candidates"]
ats = ATSParser().parse("data/JD.txt")
ranked = KeywordRanker().rank(jd_text, core, ats)

resume_data = ResumeLoader("data/resume.json").load()
matcher = ResumeMatcher()
decision = matcher.create_decision_plan(ranked, resume_data)

runner = ResumeRewriter()
result = runner.rewrite_resume(decision, resume_data)  # runs model -> validate -> returns JSON
print(result)

c:\Users\Ayush\Desktop\model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.48s/it]
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
You are not running the flash-attention implementation, expect numerical differences.


KeyboardInterrupt: 

In [1]:
import importlib

packages = [
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "sentencepiece",
    "protobuf",
    "safetensors"
]

for pkg in packages:
    try:
        module = importlib.import_module(pkg)
        version = getattr(module, "__version__", "Version attribute not found")
        print(f"{pkg}: {version}")
    except ImportError:
        print(f"{pkg}: NOT INSTALLED")

torch: 2.1.2+cu121


c:\Users\Ayush\Desktop\model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 4.46.2
accelerate: 0.27.2
bitsandbytes: 0.43.3
sentencepiece: NOT INSTALLED
protobuf: NOT INSTALLED
safetensors: 0.7.0
